# Scikit-Learn Pipeline

In [37]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn import set_config
set_config(display="diagram")
from sklearn.pipeline import make_pipeline, make_union
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer, make_column_transformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

https://pythonsimplified.com/how-to-use-k-fold-cv-and-gridsearchcv-with-sklearn-pipeline/

## Chargement des données

In [38]:
df = pd.read_csv("sample/train.csv")   

## Séparation train et test

In [39]:
X = df.drop(['Survived'], axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33)

## Analyse des colonnes de X

In [40]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Pclass       891 non-null    int64  
 2   Name         891 non-null    object 
 3   Sex          891 non-null    object 
 4   Age          714 non-null    float64
 5   SibSp        891 non-null    int64  
 6   Parch        891 non-null    int64  
 7   Ticket       891 non-null    object 
 8   Fare         891 non-null    float64
 9   Cabin        204 non-null    object 
 10  Embarked     889 non-null    object 
dtypes: float64(2), int64(4), object(5)
memory usage: 76.7+ KB


##  Mise en place des pipelines numérique et catégoriel

In [41]:
X.columns.tolist()

['PassengerId',
 'Pclass',
 'Name',
 'Sex',
 'Age',
 'SibSp',
 'Parch',
 'Ticket',
 'Fare',
 'Cabin',
 'Embarked']

In [42]:
X.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [43]:
numeric_cols = ['Age', 'Fare', 'SibSp', 'Parch']
categorical_cols = ['Embarked', 'Sex', 'Pclass']
removed_cols = ['Name', 'Cabin', 'Ticket', 'PassengerId']

numeric_transformer = make_pipeline(
    SimpleImputer(strategy='mean'),
    StandardScaler()
)
categorical_transformer = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(handle_unknown='ignore')
)
preprocessor = make_column_transformer(
    (numeric_transformer, numeric_cols),
    ( categorical_transformer, categorical_cols), 
    ('drop', removed_cols)
)
pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clf', RandomForestClassifier())
])
pipe.fit(X_train, y_train);
y_pred = pipe.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.8135593220338984


## Trouver les meilleurs Hyper-Paramètres

In [44]:
from sklearn.model_selection import GridSearchCV

params={
    'clf__n_estimators':[100, 200, 500],
    'clf__max_depth': [5, 6, 7, 8], 
    'clf__min_samples_leaf': [2, 6, 10]
}
#For this example, you need to append clf__ to the parameter name. The clf__ is the name we had given to the estimator.
grid_pipe = GridSearchCV(pipe,
                         param_grid=params,
                         cv=5,
                         verbose=1)

grid_pipe.fit(X_train, y_train)
print(grid_pipe.best_params_)
print(grid_pipe.best_score_)

Fitting 5 folds for each of 36 candidates, totalling 180 fits
{'clf__max_depth': 8, 'clf__min_samples_leaf': 2, 'clf__n_estimators': 500}
0.8306162464985996


## RandomForest avec les meilleurs hyperparamètres

In [48]:
numeric_cols = ['Age', 'Fare', 'SibSp', 'Parch']
categorical_cols = ['Embarked', 'Sex', 'Pclass']
removed_cols = ['Name', 'Cabin', 'Ticket', 'PassengerId']

numeric_transformer = make_pipeline(
    SimpleImputer(strategy='mean'),
    StandardScaler()
)
categorical_transformer = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(handle_unknown='ignore')
)
preprocessor = make_column_transformer(
    (numeric_transformer, numeric_cols),
    ( categorical_transformer, categorical_cols), 
    ('drop', removed_cols)
)

best_params={
    'n_estimators':[500],
    'max_depth': [8], 
    'min_samples_leaf': [2]
}

pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=200,max_depth=7,min_samples_leaf=2))
])
pipe

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('pipeline-1',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer()),
                                                                  ('standardscaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare', 'SibSp',
                                                   'Parch']),
                                                 ('pipeline-2',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Embarked', 'Sex',
                                                   'Pclass']),
                                                 ('drop', 'drop',
                                                  ['Name', 'Cabin', 'Ticket',
                                                   'PassengerId'])])),
                ('clf',
                 RandomForestClassifier(max_depth=7, min_samples_leaf=2,
                                        n_estimators=200))])

In [49]:
pipe.fit(X_train, y_train);
y_pred = pipe.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.8372881355932204


In [50]:
submission = pd.DataFrame({
    "PassengerId": X_test['PassengerId'],
    "Survived": y_pred
})

submission.to_csv('sample/submission.csv', index=False)

In [51]:
submission.head()

,PassengerId,Survived
120,121,0
505,506,1
494,495,0
608,609,1
90,91,0
